In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import RandomOverSampler
import ta

# Cargar dataset original
df = pd.read_csv('SPY.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Asignar ID único para poder recuperar filas luego del oversampling
df['row_id'] = df.index

# Crear indicadores técnicos
df['SMA_20'] = df['Adj Close'].rolling(window=20).mean()
df['SMA_50'] = df['Adj Close'].rolling(window=50).mean()
df['daily_return'] = df['Adj Close'].pct_change()
df['RSI_14'] = ta.momentum.RSIIndicator(close=df['Adj Close'], window=14).rsi()
macd = ta.trend.MACD(close=df['Adj Close'])
df['MACD'] = macd.macd()
df['Signal_Line'] = macd.macd_signal()
df['target'] = np.where(df['Adj Close'].shift(-1) > df['Adj Close'], 1, 0)
df['momentum_5d'] = df['Adj Close'] - df['Adj Close'].shift(5)
df['return_3d'] = df['daily_return'].rolling(window=3).sum()
df['volume_change'] = df['Volume'].pct_change()
threshold = 3 * df['daily_return'].std()
df['outlier_return'] = (np.abs(df['daily_return']) > threshold).astype(int)

# Normalización
scaler = MinMaxScaler()
df[['Adj Close_norm', 'Volume_norm']] = scaler.fit_transform(df[['Adj Close', 'Volume']])

# Features nuevas
new_cols = [
    'SMA_20', 'SMA_50', 'daily_return', 'RSI_14', 'MACD', 'Signal_Line',
    'momentum_5d', 'return_3d', 'volume_change', 'outlier_return',
    'Adj Close_norm', 'Volume_norm'
]

# Banderas de imputación
for col in new_cols:
    df[f'{col}_was_imputed'] = df[col].isna().astype(int)

# Imputaciones inteligentes
df['SMA_20'] = df['SMA_20'].fillna(method='bfill')
df['SMA_50'] = df['SMA_50'].fillna(method='bfill')
df['daily_return'] = df['daily_return'].fillna(0)
df['RSI_14'] = df['RSI_14'].fillna(method='bfill')
df['MACD'] = df['MACD'].fillna(method='bfill')
df['Signal_Line'] = df['Signal_Line'].fillna(method='bfill')
df['momentum_5d'] = df['momentum_5d'].fillna(0)
df['return_3d'] = df['return_3d'].fillna(0)
df['volume_change'] = df['volume_change'].fillna(0)
df['outlier_return'] = df['outlier_return'].fillna(0)
df['Adj Close_norm'] = df['Adj Close_norm'].fillna(method='bfill')
df['Volume_norm'] = df['Volume_norm'].fillna(method='bfill')
df['target'] = df['target'].fillna(method='ffill').astype(int)

# Usamos row_id como referencia para recuperar datos originales
X = df[new_cols + [f'{col}_was_imputed' for col in new_cols] + ['row_id']]
y = df['target']

# Oversampling con RandomOverSampler
ros = RandomOverSampler(sampling_strategy='minority', random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)

# Recuperar filas originales con row_id
df_balanced = df.loc[X_resampled['row_id']].copy().reset_index(drop=True)
df_balanced['target'] = y_resampled  # aseguramos target correcto post-oversampling
df_balanced.drop(columns=['row_id'], inplace=True)

# Guardar resultado
df_balanced.to_csv('SPY_balanced_all_columns.csv', index=False)
print("✅ Dataset completo balanceado y sin errores guardado como 'SPY_balanced_all_columns.csv'")


✅ Dataset completo balanceado y sin errores guardado como 'SPY_balanced_all_columns.csv'


/tmp/ipykernel_701/962613356.py:18: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['daily_return'] = df['Adj Close'].pct_change()
/tmp/ipykernel_701/962613356.py:26: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['volume_change'] = df['Volume'].pct_change()
/tmp/ipykernel_701/962613356.py:46: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['SMA_20'] = df['SMA_20'].fillna(method='bfill')
/tmp/ipykernel_701/962613356.py:47: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a futur